# Residuals: Visual Diagnostics and Numerical Diagnostics

**Helpful Resource:**
- [Python Reference](http://data8.org/sp22/python-reference.html): Cheat sheet of helpful array & table methods used in Data 8!

**Recommended Readings**: 
- [Visual Diagnoistics](https://inferentialthinking.com/chapters/15/5/visual-diagnostics/)
- [Numerical Diagnoistics](https://inferentialthinking.com/chapters/15/6/numerical-diagnostics/)

In [ ]:
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
from math import ceil

In [ ]:
def standard_units(numbers_array):
    "Convert any array of numbers to standard units."
    return (numbers_array - np.mean(numbers_array))/np.std(numbers_array)

def correlation(table, label_x, label_y):
    return np.mean(standard_units(table.column(label_x))*standard_units(table.column(label_y)))

def slope(table, label_x, label_y):
    r = correlation(table, label_x, label_y)
    return r*np.std(table.column(label_y))/np.std(table.column(label_x))

def intercept(table, label_x, label_y):
    return np.mean(table.column(label_y)) - slope(table, label_x, label_y)*np.mean(table.column(label_x))
    
def fitted_values(table, x, y):
    """Return an array of the regression estimates (predictions) at all the x value."""
    a = slope(table, x, y)
    b = intercept(table, x, y)
    return a * table.column(x) + b

def residuals(table, x, y):
    predictions = fitted_values(table, x, y)
    return table.column(y) - predictions

### Residuals

Residuals are the vertical distances of the points from the regression line. There is one residual for each point in the scatter plot.

residual = observed value − regression estimate

residual = y − fitted value of y

residual = y − height of regression line at x

In other words, residuals are errors.

<img src="rmse-a.png" alt="root mean square error" width=400/>

The vertical red lines from the points to the regression line are residuals.

### Residual - Positive, Negative and Zero

If a data point is **above** the regression line, it has a **positive residual**, and the regression line underestimated the true value.

If a data point is **below** the regression line, it has a **negative residual**, and the regression line overestimated the true value.

If a data point is **on** the regression line, it has a **zero residual**, and the regression line estimated the true value correctly.

### Residuel Plots Show No Trend

**"For every linear regression (least-squares), whether good or bad, the residual plot shows no trend. Overall, it is flat. In other words, the residuals and the predictor variable are uncorrelated."** 

The correlation coefficient ($r$) of residual plots are very close to a 0. In other words, in Python, floating-point rounding makes it extremely close to 0, effectively 0.

### Average of Residuals

**"No matter what the shape of the scatter diagram, the average of the residuals is 0."**

### SD of the Residuals

No matter what the shape of the scatter plot, the SD of the residuals is a fraction of the SD of the response variable. The fraction is $\sqrt{1 - r^2}$.

$\text{SD of residuals} = \sqrt{1 - r^2} \cdot \text{SD of y}$





### Regression Diagnostics

Residual plots help us make visual assessments of the quality of a linear regression analysis. Such assessments are called diagnostics. 

In [ ]:
salaries = Table.read_table('experience_vs_compensation.csv').drop('Engineer_ID')
# remove the oulliers
salaries = salaries.where('Total_Compensation_USD', are.below(300001))
salaries.scatter('Experience_Years', 'Total_Compensation_USD')

In [ ]:
# check the r
correlation(salaries, 'Experience_Years', 'Total_Compensation_USD')

In [ ]:
# the Fitted Value is the regression line estimation
salaries = salaries.with_columns(
        'Fitted Value', fitted_values(salaries, 'Experience_Years', 'Total_Compensation_USD'),
        'Residual', residuals(salaries, 'Experience_Years', 'Total_Compensation_USD')
    )
salaries

In [ ]:
# draw a scatter plot with the regression line estimation
def scatter_fit(table, x, y):
    table.scatter(x, y, s=15)
    plots.plot(table.column(x), fitted_values(table, x, y), lw=4, color='gold')
    plots.xlabel(x)
    plots.ylabel(y)

In [ ]:
scatter_fit(salaries, 'Experience_Years', 'Total_Compensation_USD')

In [ ]:
# draw a residual plot
def residual_plot(table, x, y):
    x_array = table.column(x)
    t = Table().with_columns(
            x, x_array,
            'residuals', residuals(table, x, y)
        )
    t.scatter(x, 'residuals', color='r')
    xlims = make_array(min(x_array), max(x_array))
    plots.plot(xlims, make_array(0, 0), color='darkblue', lw=4)
    plots.title('Residual Plot')

In [ ]:
residual_plot(salaries, 'Experience_Years', 'Total_Compensation_USD')

#### The residuals and the predictor variable are uncorrelated.

$r = 0$

In other words, in Python, floating-point rounding makes it extremely close to 0, effectively 0.

In [ ]:
# correlation between the predictor x and the residuals is very close to Zero
round(correlation(salaries, 'Experience_Years', 'Residual'), 4)

The accuracy of the regression appears to be about the same across the observed range of the predictor variable.

Other than a few outliers, the plot isn’t narrower in some places and wider in others.

**"The residual plot of a good regression shows no pattern. The residuals look about the same, above and below the horizontal line at 0, across the range of the predictor variable."**

### Detecting Nonlinearity - trend to have a curve shape

#### Example 1

In [ ]:
cars = Table().read_table('Cars2015_v1.csv')
#cars.show(3)

In [ ]:
cars.scatter('Height', 'FuelCap') 

In [ ]:
# check the r
correlation(cars, 'Height', 'FuelCap')

In [ ]:
# the Fitted Value is the regression line estimation
cars_tmp = cars.select('Height', 'FuelCap').with_columns(
        'Fitted Value', fitted_values(cars, 'Height', 'FuelCap'),
        'Residual', residuals(cars, 'Height', 'FuelCap')
    )
cars_tmp.show(3)

In [ ]:
# draw a scatter plot with the regression line estimation
scatter_fit(cars_tmp, 'Height', 'FuelCap')

In [ ]:
# draw a residual plot
residual_plot(cars_tmp, 'Height', 'FuelCap')

##### To repeat, the residuals and the predictor variable are uncorrelated.

$r = 0$

In other words, in Python, floating-point rounding makes it extremely close to 0, effectively 0.

In [ ]:
# correlation between the predictor x and the residuals is very close to Zero
round(correlation(cars_tmp, 'Height', 'Residual'), 4)

#### Example 2

Dugongs are marine mammals.

Let's estimate/predict the age of dugongs based on their length.

In [ ]:
dugong = Table.read_table('dugong.csv')
dugong.show(5)

In [ ]:
# check the r
correlation(dugong, 'Length', 'Age')

In [ ]:
# the Fitted Value is the regression line estimation
dugong = dugong.with_columns(
        'Fitted Value', fitted_values(dugong, 'Length', 'Age'),
        'Residual', residuals(dugong, 'Length', 'Age')
    )
dugong.show(3)

In [ ]:
# draw a scatter plot with the regression line estimation
scatter_fit(dugong, 'Length', 'Age')

In [ ]:
# draw a residual plot
residual_plot(dugong, 'Length', 'Age')

##### Once again, the residuals and the predictor variable are uncorrelated.

$r = 0$

In Python, floating-point rounding makes it extremely close to 0, effectively 0.

In [ ]:
# correlation between the predictor x and the residuals is very close to Zero
round(correlation(dugong, 'Length', 'Residual'), 4)

**"When a residual plot shows a pattern, there may be a non-linear relation between the variables."**

### Detecting Heteroscedasticity

Heteroscedasticity in data science means "“uneven spread”.

_Too hard for me to pronounce right..._

#### Example 1

In [ ]:
cars.scatter('LowPrice', 'UTurn')

In [ ]:
correlation(cars, 'LowPrice', 'UTurn')

In [ ]:
# Fitted Value is the regression line estimation
cars_tmp = cars.select('LowPrice', 'UTurn').with_columns(
        'Fitted Value', fitted_values(cars, 'LowPrice', 'UTurn'),
        'Residual', residuals(cars, 'LowPrice', 'UTurn')
    )
cars_tmp.show(3)

In [ ]:
# check out the regression line the scatter plot
scatter_fit(cars_tmp, 'LowPrice', 'UTurn')

In [ ]:
# draw a residual plot
residual_plot(cars_tmp, 'LowPrice', 'UTurn')

##### The residuals and the predictor variable are uncorrelated.

$r = 0$

In [ ]:
# correlation between the predictor x and the residuals is very close to Zero
round(correlation(cars_tmp, 'LowPrice', 'Residual'), 4)

#### Example 2

In [ ]:
hybrid = Table.read_table('hybrid.csv')
hybrid.show(3)

In [ ]:
hybrid.scatter('acceleration', 'mpg')

In [ ]:
# check the r
correlation(hybrid, 'acceleration', 'mpg')

In [ ]:
# Fitted Value is the regression line estimation
hybrid_tmp = hybrid.select('acceleration', 'mpg').with_columns(
        'Fitted Value', fitted_values(hybrid, 'acceleration', 'mpg'),
        'Residual', residuals(hybrid, 'acceleration', 'mpg')
    )
hybrid_tmp.show(3)

In [ ]:
# draw the regression line on the scatter plot
scatter_fit(hybrid_tmp, 'acceleration', 'mpg')

In [ ]:
# draw a residual plot
residual_plot(hybrid_tmp, 'acceleration', 'mpg')

##### The residuals and the predictor variable are uncorrelated.

$r = 0$

In [ ]:
# correlation between the predictor x and the residuals is very close to Zero
round(correlation(hybrid_tmp, 'acceleration', 'Residual'), 4)

**"If the residual plot shows uneven variation about the horizontal line at 0, the regression estimates are not equally accurate across the range of the predictor variable."**